# ※ 과제 안내

- **과제 배점**: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.

- **채점 기준**:
    - 출력 결과 일치: 제출한 코드가 제시된 출력 결과 또는 채점 기준과 일치하는 경우에만 정답으로 인정됩니다.
    - 코드의 다양성 인정: 출력 결과가 동일하다면 다양한 접근 방식도 정답으로 인정됩니다.

- **킬러 문항 안내** (문제 3, 6, 7, 9, 10):
    - 코드 작성만으로는 채점되지 않습니다.
    - **실행 결과 스크린샷** 또는 **정량 수치가 포함된 출력 캡처**를 반드시 함께 제출해야 합니다.
    - 스크린샷 없이 코드만 제출 시 0점 처리됩니다.

---
# RAG 시스템 개발 과제
### Part 1 ~ Part 7 종합

---
## 공통 환경 설정
> 모든 문제 실행 전 아래 두 셀을 먼저 실행하세요.

In [ ]:
# 패키지 설치 (최초 1회)
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters
!pip install -q chromadb sentence-transformers pdfplumber python-docx beautifulsoup4 requests
!pip install -q ragas datasets

In [ ]:
# Google Gemini API 키 설정
# API 키 발급: https://aistudio.google.com/apikey (Google 계정으로 무료 발급)
#
# 이 과제는 Google Gemini API를 사용합니다.
#    - LLM 모델: gemini-3.1-flash-lite-preview
#      → Gemini 3 계열 최신 경량 모델. 무료 티어에서 사용 가능.
#    - 임베딩 모델: gemini-embedding-001 (무료 티어 지원, MTEB 멀티링궐 리더보드 상위권)
#    OpenAI 대비 무료 티어 일일 요청 한도가 높아 과제 전체 수행에 적합합니다.
#    (참고: 유사 모델 기준 무료 티어 약 1,000 요청/일)

import os
os.environ["GOOGLE_API_KEY"] = "your-google-api-key-here"  # 본인 API 키 입력

---
# 문제 1. 임베딩 생성 및 코사인 유사도 계산 [Part 1-2]

아래 조건을 만족하도록 코드를 완성하시오.

- `sentence-transformers`의 `all-MiniLM-L6-v2` 모델을 사용하여 주어진 문장들의 임베딩을 생성하시오.
- 쿼리 문장과 각 후보 문장 간의 코사인 유사도를 **직접 계산**하시오. (`sklearn` 등 유사도 함수 직접 호출 금지, `numpy`만 사용)
- 유사도가 높은 순서대로 순위를 출력하시오.

**채점 기준**: 각 유사도 수치가 소수점 4자리 기준 ±0.005 이내로 일치할 것

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 주어진 데이터 (수정하지 마시오)
query = "How do I request annual leave?"
candidates = [
    "Vacation request procedure and approval process",
    "Salary payment schedule and payroll information",
    "Employee benefits overview and insurance coverage",
    "Python programming tutorial for beginners",
    "How to apply for time off and holiday entitlement"
]

# TODO 1: 모델 로드 (all-MiniLM-L6-v2)
model = 

# TODO 2: query와 candidates 임베딩 생성
query_embedding = 
candidate_embeddings = 

# TODO 3: 코사인 유사도 계산 함수 구현 (numpy만 사용)
def cosine_similarity(vec_a, vec_b):
    # TODO
    pass

# TODO 4: 유사도 계산 후 높은 순서대로 출력
# 출력 형식:
# 1위 (similarity: 0.8214): Vacation request procedure and approval process
# 2위 (similarity: 0.7891): How to apply for time off and holiday entitlement
# ...

---
# 문제 2. 파일 형식 분기 파싱 파이프라인 구현 [Part 2-1]

아래 조건을 만족하는 **범용 문서 파싱 함수** `parse_document()`를 구현하시오.

- **입력**: 파일 경로 (`str`)
- **출력**: `List[Document]` (LangChain Document 객체 리스트)
- **지원 형식 및 처리 조건**:

| 형식 | 라이브러리 | 처리 조건 |
|------|-----------|----------|
| `.pdf` | `pdfplumber` | 페이지당 추출 텍스트 10자 미만 → `page_content`에 `"[SCAN_PDF_DETECTED] OCR required"` 저장 |
| `.docx` | `python-docx` | 빈 단락 건너뜀. `metadata`에 `heading_level` 포함 (일반 단락은 `0`) |
| `.html` / `.htm` | `BeautifulSoup` | `<p>`, `<h1>`~`<h3>` 텍스트만 추출. `<script>`, `<style>` 태그 내용 제외 |
| `.txt` | 내장 `open()` | 전체 내용을 Document 1개로 반환 |

- 지원하지 않는 형식은 `ValueError` 발생
- 모든 Document의 `metadata`에 `source` (파일명), `format` (확장자) 포함

**채점 기준**: 아래 테스트 코드 실행 시 `.txt`, `.html`, `.docx` 3가지 형식 모두 `List[Document]` 정상 반환 및 `metadata` 키(`source`, `format`) 포함 확인

In [ ]:
# 파싱 테스트용 샘플 파일 생성 (수정하지 마시오)
# NOTE: txt, html, docx 3종 샘플 파일을 생성합니다.
#       PDF 파싱 로직은 parse_document() 함수 내에 구현하되,
#       별도 샘플 PDF 파일은 필요하지 않습니다 (구현 로직만 작성).
import os
from docx import Document as DocxDocument
from langchain_core.documents import Document

os.makedirs("sample_docs", exist_ok=True)

# --- TXT 파일 (노이즈 포함: 반복 헤더/푸터, 하이픈 줄바꿈, 이중 공백) ---
txt_content = """INTERNAL KNOWLEDGE BASE DOCUMENT
Organization: TechCorp AI Division
Document ID: KB-2024-0042
Last Updated: March 2024
Classification: Internal Use Only
--------------------------------------------------

SECTION 1: INTRODUCTION TO RETRIEVAL-AUGMENTED GENERATION

Retrieval-Augmented Generation (RAG) is an AI architecture  that com-
bines the strengths of large language models with external knowledge re-
trieval systems. Unlike traditional LLMs, which rely solely on parametric
memory acquired during pre-training, RAG systems dynamically fetch rele-
vant information from a curated knowledge base at inference time.

The fundamental problem RAG addresses is the knowledge cutoff limitation.
LLMs are trained on static datasets with a fixed cutoff date, making them
unable to answer questions about recent events or proprietary information.
Additionally, LLMs are prone to hallucination — generating factually  in-
correct but linguistically plausible responses. RAG mitigates both issues
by grounding model outputs in retrieved, verifiable source documents.

--------------------------------------------------
Page 1 of 3  |  TechCorp Confidential
--------------------------------------------------

SECTION 2: CORE PIPELINE STAGES

A RAG pipeline consists of two primary phases: Indexing (offline) and
Querying (online). During Indexing, documents are collected, parsed,
pre-processed, chunked, embedded, and stored in a vector database.
During Querying, a user's question is embedded and used to retrieve the
most relevant chunks, which are then passed as context to the LLM.

2.1  Document Parsing
Documents arrive in heterogeneous formats: PDF, DOCX, HTML, CSV, and
plain text. Each format requires a dedicated parsing strategy. PDF  files
are particularly challenging because they store text as layout coordinates
rather than sequential strings, making multi-column and scanned documents
difficult to handle correctly. DOCX files, by contrast, expose an explicit
XML structure, enabling heading-level extraction and table parsing.

2.2  Text Chunking
Chunking divides long documents into smaller units suitable for embedding.
The two key hyperparameters are chunk_size (maximum token count per chunk)
and chunk_overlap (shared tokens between adjacent chunks). Smaller chunks
yield higher retrieval precision but may lose surrounding context. Larger
chunks preserve context but dilute the embedding signal. Empirical guidance
suggests starting with chunk_size=400 and overlap=10-20% of chunk_size,
then tuning based on Recall@k evaluation results.

--------------------------------------------------
Page 2 of 3  |  TechCorp Confidential
--------------------------------------------------

SECTION 3: VECTOR DATABASES AND RETRIEVAL

Embeddings generated by models such as gemini-embedding-001 or
sentence-transformers/all-MiniLM-L6-v2 are stored in vector databases
optimised for Approximate Nearest Neighbour (ANN) search. Popular options
include ChromaDB (lightweight, Python-native), FAISS (high-performance,
Facebook AI), and Qdrant (cloud-native, Rust-based). Retrieval is performed
by computing cosine similarity between the query embedding and all indexed
document vectors, returning the top-k most similar chunks.

Re-ranking further refines retrieval quality. A Cross-Encoder model  such
as cross-encoder/ms-marco-MiniLM-L-6-v2 scores each (query, document) pair
jointly, capturing fine-grained relevance signals that Bi-Encoders miss.
This 2-Stage approach (Bi-Encoder → Top-K candidates → Cross-Encoder → re-
ranked Top-N) balances speed and accuracy in production systems.

--------------------------------------------------
Page 3 of 3  |  TechCorp Confidential
--------------------------------------------------
"""
with open("sample_docs/sample.txt", "w", encoding="utf-8") as f:
    f.write(txt_content)

# --- HTML 파일 (노이즈 포함: <nav>, <footer>, <script>, <style>, 광고 배너) ---
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>RAG Systems – Developer Guide</title>
  <style>
    body { font-family: Arial, sans-serif; color: #333; }
    .ad-banner { background: yellow; }
  </style>
</head>
<body>
  <nav>
    <a href="/home">Home</a> | <a href="/docs">Docs</a> | <a href="/api">API</a>
  </nav>

  <div class="ad-banner">SPONSORED: Try CloudVector DB — 3 months free!</div>

  <h1>RAG Systems: A Developer Guide</h1>
  <p>Retrieval-Augmented Generation (RAG) represents a significant leap in how
  AI systems handle knowledge-intensive tasks. By decoupling knowledge storage
  from model parameters, RAG enables organisations to update their AI knowledge
  base without expensive model re-training cycles.</p>

  <h2>Why RAG Outperforms Fine-Tuning for Dynamic Knowledge</h2>
  <p>Fine-tuning embeds knowledge into model weights, which is costly and
  brittle — any update requires a full re-training run. RAG, by contrast,
  stores knowledge externally in a vector database that can be updated in
  real time. This makes RAG the preferred architecture for enterprise
  applications with frequently changing knowledge bases, such as legal
  databases, product documentation, and customer support systems.</p>

  <h2>Core Components of a RAG Pipeline</h2>
  <p>A production RAG system comprises the following components:
  (1) a document ingestion layer that handles diverse file formats;
  (2) a text processing module responsible for cleaning, normalisation,
  and chunking; (3) an embedding service that converts text chunks into
  dense vector representations; (4) a vector store for efficient similarity
  search; and (5) a generation layer where an LLM synthesises an answer
  from the retrieved context.</p>

  <h3>Document Ingestion</h3>
  <p>The ingestion layer must handle PDF (both text-based and scanned),
  DOCX, HTML, Markdown, CSV, and plain text. LangChain's document loader
  abstractions provide a unified <code>Document</code> interface with
  <code>page_content</code> and <code>metadata</code> fields, decoupling
  downstream processing from format-specific parsing logic.</p>

  <h3>Embedding and Vector Storage</h3>
  <p>Embedding models map text into high-dimensional vector spaces where
  semantically similar texts are geometrically close. The cosine similarity
  metric — the dot product of two unit vectors — is the standard measure
  of semantic distance in this space. Models like gemini-embedding-001
  produce 3072-dimensional embeddings and rank among the top performers
  on the MTEB multilingual benchmark.</p>

  <h3>Re-ranking and Answer Generation</h3>
  <p>Initial retrieval using a Bi-Encoder is fast but imprecise. A
  Cross-Encoder re-ranker refines the candidate set by scoring each
  (query, passage) pair jointly, dramatically improving precision at
  the cost of higher latency. The re-ranked passages are concatenated
  into a prompt and passed to the generation model alongside the
  original user question.</p>

  <script>
    // Analytics tracking — should be excluded from parsed content
    window.analytics = { pageView: 'rag-developer-guide' };
    console.log('Page loaded:', window.analytics);
  </script>

  <footer>
    &copy; 2024 TechCorp AI Division. All rights reserved.
    | <a href="/privacy">Privacy Policy</a>
    | <a href="/terms">Terms of Use</a>
  </footer>
</body>
</html>
"""
with open("sample_docs/sample.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# --- DOCX 파일 (다단계 헤더, 의도적 빈 단락 포함) ---
doc = DocxDocument()
doc.add_heading('Technical Reference: RAG System Architecture', level=1)
doc.add_paragraph('')  # intentional empty paragraph
doc.add_paragraph(
    'This document provides a technical overview of Retrieval-Augmented '
    'Generation (RAG) systems, intended for ML engineers onboarding to '
    'the TechCorp AI platform. It covers pipeline design, component '
    'interfaces, and deployment considerations.'
)
doc.add_heading('1. System Overview', level=2)
doc.add_paragraph(
    'A RAG system augments a large language model with a retrieval '
    'mechanism that fetches relevant passages from an external corpus '
    'at inference time. This approach addresses two critical limitations '
    'of standard LLMs: knowledge staleness (information beyond the '
    'training cutoff) and hallucination (confident but factually '
    'incorrect generation).'
)
doc.add_paragraph('')  # intentional empty paragraph
doc.add_heading('1.1 Indexing Phase', level=3)
doc.add_paragraph(
    'During indexing, source documents are loaded, parsed into plain '
    'text, pre-processed to remove noise, split into chunks, embedded '
    'into vector representations, and stored in a vector database. '
    'This phase runs offline and is repeated whenever the knowledge '
    'base is updated.'
)
doc.add_heading('1.2 Querying Phase', level=3)
doc.add_paragraph(
    'At query time, the user\'s question is embedded using the same '
    'model used during indexing. A nearest-neighbour search retrieves '
    'the top-k most relevant chunks. These chunks are optionally '
    're-ranked by a Cross-Encoder model, then concatenated into a '
    'context prompt that is passed to the LLM along with the original '
    'question.'
)
doc.add_heading('2. Component Specifications', level=2)
doc.add_heading('2.1 Document Loader', level=3)
doc.add_paragraph(
    'The DocumentLoader component accepts a file path and returns a '
    'List[Document], where each Document contains page_content (str) '
    'and metadata (dict). Supported formats: PDF, DOCX, HTML, TXT. '
    'For scanned PDFs (< 10 chars per page), the loader must flag the '
    'document for OCR rather than returning empty content.'
)
doc.add_heading('2.2 Text Processor', level=3)
doc.add_paragraph(
    'The TextProcessor applies a RecursiveCharacterTextSplitter with '
    'configurable chunk_size and chunk_overlap. The splitter attempts '
    'to break text at paragraph boundaries first, then sentence '
    'boundaries, then word boundaries, recursively until the target '
    'chunk size is met. Metadata from the parent Document is propagated '
    'to all child chunks.'
)
doc.add_paragraph('')  # intentional empty paragraph
doc.add_heading('3. Evaluation Framework', level=2)
doc.add_paragraph(
    'RAG systems are evaluated using the Ragas framework, which provides '
    'four core metrics: Faithfulness (is the answer grounded in the '
    'retrieved context?), Answer Relevancy (does the answer address the '
    'question?), Context Precision (what fraction of retrieved chunks '
    'were actually useful?), and Context Recall (were all necessary '
    'chunks retrieved?). A Faithfulness score below 0.7 typically '
    'indicates prompt or context quality issues.'
)
doc.save("sample_docs/sample.docx")

print("Sample files generated: sample.txt, sample.html, sample.docx")
print(f"   TXT  : {len(txt_content):,} chars")
print(f"   HTML : {len(html_content):,} chars")
print(f"   DOCX : multi-level headings + intentional empty paragraphs")


In [ ]:
import pdfplumber
from docx import Document as DocxDocument
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from typing import List

# TODO: 범용 파싱 함수 구현
def parse_document(file_path: str) -> List[Document]:
    """
    Auto-detects file format and returns a list of LangChain Document objects.

    Args:
        file_path: path to the file to parse
    Returns:
        List[Document]
    Raises:
        ValueError: unsupported file format
    """
    # TODO
    pass


# 테스트 코드 (수정하지 마시오)
for path in ["sample_docs/sample.txt", "sample_docs/sample.html", "sample_docs/sample.docx"]:
    docs = parse_document(path)
    print(f"\n[{path}]")
    print(f"  Document 수: {len(docs)}")
    print(f"  첫 번째 page_content: {docs[0].page_content[:50]}...")
    print(f"  metadata: {docs[0].metadata}")
    assert "source" in docs[0].metadata, "metadata에 'source' 키 없음"
    assert "format" in docs[0].metadata, "metadata에 'format' 키 없음"

print("\n모든 테스트 통과")

---
# 문제 3. Chunking 파라미터 실험 및 전략 비교 [Part 2-3] 

동일한 샘플 문서에 대해 아래 **3가지 청킹 설정**을 적용하고 결과를 비교·분석하시오.

| 설정 | chunk_size | chunk_overlap | Splitter |
|------|-----------|---------------|----------|
| A | 200 | 0 | `RecursiveCharacterTextSplitter` |
| B | 200 | 50 | `RecursiveCharacterTextSplitter` |
| C | 500 | 100 | `RecursiveCharacterTextSplitter` |

**각 설정별 필수 출력 항목**:
1. 생성된 청크 수
2. 평균 청크 길이 (문자 수, 소수점 1자리)
3. 최소 / 최대 청크 길이
4. 쿼리(`"How to store documents in a vector database"`)에 대해 코사인 유사도 기준 Top-1 청크 내용 (앞 60자)

> 유사도 계산 시 `all-MiniLM-L6-v2` 모델 사용

**분석 질문** — 아래 마크다운 셀에 직접 작성:
- `chunk_overlap` 적용 시 청크 수가 어떻게 변하는지, 그 이유를 설명하시오.
- 3가지 설정 중 이 문서에 가장 적합한 설정을 선택하고, **수치 근거**를 포함하여 서술하시오.

**킬러 채점 기준**:
- 3가지 설정의 비교 결과 출력 **스크린샷 제출 필수**
- 분석 답변 미작성 시 0점 처리

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 실험용 문서 (수정하지 마시오)
# 주의: 이 텍스트는 의도적으로 길고 노이즈 요소(하이픈 줄바꿈, 반복 헤더,
#       불규칙 공백 등)를 포함하고 있습니다. 각 청킹 전략이 이를 어떻게
#       처리하는지 관찰하세요.
sample_text = """
===========================================================================
TECHNICAL WHITEPAPER — RAG SYSTEM DESIGN AND IMPLEMENTATION
TechCorp AI Research | Document No. WP-2024-11 | Revision 3
===========================================================================

ABSTRACT

Retrieval-Augmented Generation (RAG) is a framework that enhances large
language model (LLM) outputs by conditioning generation on dynamically
retrieved external documents. This whitepaper describes the full RAG pipe-
line, covering document ingestion, text pre-processing, chunking strategy
selection, embedding model choice, vector database architecture, retrieval
optimisation, and offline evaluation. Engineers building production RAG
systems will find actionable guidance on each stage.

---------------------------------------------------------------------------
SECTION 1: MOTIVATION AND BACKGROUND
---------------------------------------------------------------------------

1.1 The Knowledge Cutoff Problem

Large language models acquire knowledge by compressing vast corpora of text
into billions of floating-point parameters during pre-training. This process
is computationally expensive and cannot be repeated continuously. As a re-
sult, every LLM has a hard knowledge cutoff — a date beyond which it has no
awareness of world events. An LLM trained through Q3 2023 cannot answer
questions about Q1 2024 product launches, regulatory changes, or research
findings. For enterprise applications, this limitation is often unacceptable.

1.2 Hallucination and Factual Reliability

Even within its training distribution, an LLM may generate confidently-
worded but factually incorrect statements — a phenomenon termed hallucina-
tion. This occurs because the model's objective during training is to predict
likely next tokens, not to verify factual accuracy. RAG mitigates hallucina-
tion by forcing the model to ground its answer in explicitly retrieved, cit-
able source passages. If the answer cannot be supported by the retrieved con-
text, a well-prompted RAG system will acknowledge uncertainty rather than
fabricate.

---------------------------------------------------------------------------
SECTION 2: PIPELINE ARCHITECTURE
---------------------------------------------------------------------------

A RAG pipeline has two distinct temporal phases:

Phase A — Indexing (Offline / Batch)
  Step 1: Document Collection  — gather source files in PDF, DOCX, HTML, TXT
  Step 2: Parsing              — extract plain text + structured metadata
  Step 3: Pre-processing       — normalise encoding, remove noise artefacts
  Step 4: Chunking             — split text into embedding-sized fragments
  Step 5: Embedding            — encode chunks as dense vectors
  Step 6: Vector DB Ingestion  — store (vector, text, metadata) triples

Phase B — Querying (Online / Real-time)
  Step 1: Query Embedding      — encode user question with the same model
  Step 2: ANN Search           — retrieve top-k nearest chunks by cosine sim
  Step 3: Re-ranking (opt.)    — reorder candidates with a Cross-Encoder
  Step 4: Prompt Assembly      — combine retrieved context + original question
  Step 5: LLM Generation       — produce the final natural-language answer
  Step 6: Source Attribution   — return answer with provenance metadata

2.1 Document Parsing Challenges

PDF is the most prevalent enterprise document format and the most difficult
to parse reliably. Internally, PDF encodes text as (x, y, font, size) tuples
rather than ordered character sequences. A two-column academic paper may be-
come garbled when a naïve parser linearises columns left-to-right across the
page width. Header and footer text repeated on every page pollutes the corpus
with low-signal noise. Scanned PDFs contain no machine-readable text at all
and require OCR as a pre-processing step.

DOCX files are structurally richer: the Open XML schema tags paragraphs with
explicit style identifiers (Heading1, Heading2, Normal), enabling downstream
chunking strategies to respect document hierarchy. Tables are stored as XML
Row/Cell elements, preserving row-column relationships.

HTML documents from the web include large amounts of non-content text: navi-
gation menus, cookie banners, advertising scripts, and footer disclaimers.
BeautifulSoup's tag-selective extraction (targeting <p>, <h1>–<h3>) while
removing <script>, <style>, and <nav> is the standard approach, though it
requires domain-specific tuning for complex page layouts.

2.2 Chunking Strategy Selection

The optimal chunking strategy depends on document structure and retrieval
objectives. Five major approaches exist:

  (a) Fixed-size character splitting  — simple, fast, ignores semantics
  (b) Recursive character splitting   — respects paragraph/sentence breaks
  (c) Markdown/HTML header splitting  — leverages document hierarchy
  (d) Semantic chunking               — uses embedding similarity to detect
                                        topic boundaries; high quality but
                                        computationally expensive at index time
  (e) Parent-Child chunking           — stores large chunks for context but
                                        retrieves small chunks for precision

Key hyperparameters for (a) and (b): chunk_size controls the maximum fragment
length (measured in characters or tokens), and chunk_overlap controls the
number of shared characters between adjacent chunks. A chunk_overlap of 10–20%
of chunk_size is a common starting point. Reducing chunk_size below ~128 tokens
tends to hurt recall; exceeding ~1024 tokens tends to dilute embedding quality.

---------------------------------------------------------------------------
SECTION 3: EMBEDDING MODELS AND VECTOR DATABASES
---------------------------------------------------------------------------

3.1 Choosing an Embedding Model

Embedding model selection has a larger impact on retrieval quality than
almost any other hyperparameter. Key criteria include:

  - Dimensionality: Higher-dimensional embeddings capture richer semantics
    but increase storage and query latency. Common choices: 384-d (MiniLM),
    768-d (multilingual models), 1536-d (OpenAI ada-002), 3072-d (Gemini).
  - Max input tokens: Most models cap at 512 tokens; exceeding this truncates
    the input silently, degrading quality for long chunks.
  - Language coverage: Multilingual models are essential for non-English or
    mixed-language corpora.
  - Benchmark performance: The MTEB (Massive Text Embedding Benchmark) pro-
    vides standardised retrieval task scores. Gemini-embedding-001 consistently
    ranks in the top tier on the MTEB multilingual leaderboard.
  - Asymmetric vs symmetric: Some use cases (question→passage) benefit from
    asymmetric models that use separate encoders for queries and documents.

3.2 Vector Database Options

Storing and querying millions of high-dimensional vectors requires purpose-
built data structures. The main options are:

  ChromaDB   — open-source, in-process or client-server, ideal for prototypes
               and small-scale production (< 10M vectors)
  FAISS      — Facebook AI Similarity Search; extremely fast, CPU/GPU support,
               no built-in persistence layer; best for offline batch retrieval
  Qdrant     — cloud-native, Rust-based, production-grade filtering, supports
               named vectors and payload indexing for hybrid search
  Weaviate   — GraphQL interface, built-in BM25, supports multi-tenancy
  Pinecone   — fully managed, serverless, strong Python SDK

For the exercises in this assignment, ChromaDB's ephemeral (in-memory) client
is used to avoid infrastructure dependencies.

---------------------------------------------------------------------------
SECTION 4: RETRIEVAL OPTIMISATION AND RE-RANKING
---------------------------------------------------------------------------

4.1 Bi-Encoder Retrieval (Stage 1)

A Bi-Encoder embeds the query and each document independently, then measures
similarity via dot product or cosine similarity. Because embeddings are pre-
computed at index time, Stage 1 retrieval is extremely fast — O(1) per query
with ANN indices such as HNSW (Hierarchical Navigable Small World). However,
the independent encoding means the model cannot model query-document interac-
tions, which limits precision at the top of the ranking.

4.2 Cross-Encoder Re-ranking (Stage 2)

A Cross-Encoder processes the query and a candidate document jointly, produc-
ing a scalar relevance score. The joint encoding captures token-level cross-
attention between query and document, dramatically improving precision. The
trade-off is latency: a Cross-Encoder must run N forward passes for N candi-
dates and cannot be pre-computed. In practice, Stage 2 is applied only to the
top-K candidates (K = 20–50) retrieved by Stage 1.

4.3 Measuring Retrieval Quality

Recall@k = (queries where gold document is in top-k results) / (total queries)

A Recall@5 of 0.8 or above is generally considered acceptable for production
RAG systems. score_threshold filtering removes low-confidence results below a
cosine-distance cutoff, reducing noise at the cost of potentially lower recall.

---------------------------------------------------------------------------
END OF DOCUMENT
TechCorp Confidential — Not for External Distribution
===========================================================================
"""

query = "How to store documents in a vector database"

# 임베딩 모델 로드 (sentence-transformers 사용, API 키 불필요)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 실험 설정 (수정하지 마시오)
configs = [
    {"name": "Config A", "chunk_size": 200, "chunk_overlap": 0},
    {"name": "Config B", "chunk_size": 200, "chunk_overlap": 50},
    {"name": "Config C", "chunk_size": 500, "chunk_overlap": 100},
]

for config in configs:
    print(f"\n{'='*55}")
    print(f"[{config["name"]}] chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']}")
    print("="*55)

    # TODO 1: RecursiveCharacterTextSplitter 생성 및 청킹 수행
    splitter = 
    chunks = 

    # TODO 2: 청크 수, 평균/최소/최대 길이 계산 및 출력

    # TODO 3: 쿼리와 가장 유사한 청크 Top-1 출력 (코사인 유사도 직접 계산)
    # 출력 형식:
    # 청크 수: N개 | 평균 길이: XX.X자 | 최소: XXX자 | 최대: XXX자
    # Top-1 청크: "...내용 앞 60자..."

### 📝 분석 답변 (직접 작성 — 미작성 시 0점)

**Q1. `chunk_overlap` 적용 시 청크 수 변화 및 그 이유:**

(여기에 작성)

**Q2. 가장 적합한 설정 선택 및 수치 근거:**

(여기에 작성)

---
# 문제 4. 임베딩 모델 2종 성능 비교 [Part 3-1]

아래 조건을 만족하도록 코드를 완성하시오.

- **모델 A**: `all-MiniLM-L6-v2` (sentence-transformers)
- **모델 B**: `paraphrase-multilingual-MiniLM-L12-v2` (sentence-transformers)
- 동일한 쿼리에 대해 두 모델의 유사도 Top-3 결과를 비교하시오.
- 각 모델의 **임베딩 차원(dimension)**도 함께 출력하시오.

**채점 기준**:
- 두 모델 각각의 임베딩 차원 출력
- Top-3 순위 및 유사도 수치 출력 (소수점 4자리)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 주어진 데이터 (수정하지 마시오)
query = "What is the process for submitting an expense report?"
candidates = [
    "How to file a reimbursement request for business expenses",
    "Employee salary and compensation structure overview",
    "Annual leave and vacation request submission process",
    "Step-by-step guide to expense claim submission and approval",
    "Office supply procurement and purchase order process",
    "Performance review and appraisal scheduling procedure",
]

model_names = [
    "all-MiniLM-L6-v2",
    "paraphrase-multilingual-MiniLM-L12-v2"
]

for model_name in model_names:
    # TODO 1: 모델 로드
    model = 

    # TODO 2: query 및 candidates 임베딩 생성
    query_emb = 
    cand_embs = 

    # TODO 3: 임베딩 차원 출력

    # TODO 4: 코사인 유사도 계산 후 Top-3 출력
    # 출력 형식:
    # ============================
    # [모델: all-MiniLM-L6-v2]
    # 임베딩 차원: 384
    # 1위 (0.8521): How to file a reimbursement request for business expenses
    # 2위 (0.7234): Step-by-step guide to expense claim submission and approval
    # 3위 (0.4512): Office supply procurement and purchase order process
    print("-" * 60)

---
# 문제 5. ChromaDB 벡터 DB 구축 및 유사도 검색 [Part 3-2]

아래 조건을 만족하도록 코드를 완성하시오.

- `chromadb` 라이브러리를 사용하여 **인메모리 벡터 DB**를 구축하시오.
- 주어진 문서 10개를 `all-MiniLM-L6-v2` 모델로 임베딩하여 ChromaDB에 저장하시오.
- 쿼리 2개에 대해 각각 **Top-3 검색 결과**를 출력하시오.
- 각 결과에 **document ID**, **유사도 거리(distance)**, **내용 앞 40자**를 포함하시오.

**채점 기준**: 두 쿼리에 대해 Top-3 결과 정상 반환 및 distance 수치 포함 출력

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# 저장할 문서 (수정하지 마시오)
documents = [
    {"id": "doc_01", "text": "Vector databases are purpose-built storage systems that efficiently index and query high-dimensional embedding vectors using Approximate Nearest Neighbour (ANN) algorithms."},
    {"id": "doc_02", "text": "ChromaDB is an open-source, Python-native vector database supporting both ephemeral in-memory mode for prototyping and persistent disk-based storage for production use."},
    {"id": "doc_03", "text": "Cosine similarity measures the angle between two vectors in high-dimensional space, returning a score between -1 and 1; it is the standard metric for semantic similarity search."},
    {"id": "doc_04", "text": "In a RAG pipeline, the retrieval stage embeds the user query, performs an ANN search against indexed document vectors, and returns the top-k most relevant chunks as context for the LLM."},
    {"id": "doc_05", "text": "Embedding models such as sentence-transformers/all-MiniLM-L6-v2 encode text as dense floating-point vectors, mapping semantically similar sentences to nearby points in vector space."},
    {"id": "doc_06", "text": "FAISS (Facebook AI Similarity Search) is a high-performance library for efficient similarity search over billions of vectors, with support for both CPU and GPU execution."},
    {"id": "doc_07", "text": "Qdrant is a cloud-native, Rust-based vector database offering payload filtering, named vector support, and horizontal scalability suitable for large-scale production deployments."},
    {"id": "doc_08", "text": "Chunking strategy directly determines retrieval quality: chunk_size controls semantic granularity, while chunk_overlap prevents context loss at chunk boundaries."},
    {"id": "doc_09", "text": "Metadata filtering in vector databases allows hybrid queries that combine semantic vector search with structured attribute constraints such as date range, document type, or author."},
    {"id": "doc_10", "text": "HNSW (Hierarchical Navigable Small World) is the dominant ANN index structure, offering O(log n) query time and enabling sub-millisecond nearest-neighbour retrieval over millions of vectors."},
]

queries = [
    "What are the main types and characteristics of vector databases?",
    "How does the retrieval process work in a RAG system?"
]

# TODO 1: SentenceTransformer 모델 로드 (all-MiniLM-L6-v2)
model = 

# TODO 2: ChromaDB 인메모리 클라이언트 및 컬렉션 생성
# 컬렉션 이름: "q5_collection" 고정 (다른 문제와 충돌 방지)
client = 
collection = 

# TODO 3: 문서 임베딩 후 ChromaDB에 저장
# collection.add(ids=..., embeddings=..., documents=...) 사용

# TODO 4: 각 쿼리에 대해 Top-3 검색 및 출력
for query in queries:
    # TODO
    # 출력 형식:
    # 쿼리: "What are the main types and characteristics of vector databases?"
    # Top-1 | ID: doc_XX | 거리: 0.XXXX | 내용: Vector databases are...
    # Top-2 | ID: doc_XX | 거리: 0.XXXX | 내용: ...
    # Top-3 | ID: doc_XX | 거리: 0.XXXX | 내용: ...
    pass

---
# 문제 6. 검색 파라미터 변경 실험 및 Recall@k 측정 [Part 4-1]

아래 두 가지 실험을 수행하시오.

**실험 1 — k값 변경 실험**:
- 주어진 10개 문서를 ChromaDB에 저장
- k = 1, 3, 5 각각으로 검색 수행
- 각 설정에 대해 **Recall@k** 계산:
  > `Recall@k = (Top-k 결과 안에 정답 문서가 포함된 쿼리 수) / (전체 쿼리 수)`

**실험 2 — score_threshold 실험**:
- 전체 쿼리 5개에 대해 k=10으로 검색 후 distance 기준으로 필터링
- threshold = 0.3, 0.5, 0.7 각각에서 `distance < threshold`를 만족하는 결과 수의 평균 출력

**필수 출력 형식**:
```
[실험 1: k값 변경]
k=1: Recall@1 = 0.XX (X/5)
k=3: Recall@3 = 0.XX (X/5)
k=5: Recall@5 = 0.XX (X/5)

[실험 2: score_threshold 필터링]
threshold=0.3: 평균 반환 문서 수 = X.X개
threshold=0.5: 평균 반환 문서 수 = X.X개
threshold=0.7: 평균 반환 문서 수 = X.X개
```

**⚠️ 킬러 채점 기준**:
- k=1, 3, 5에 대한 Recall@k 수치가 포함된 출력 **스크린샷 제출 필수**
- **Recall@5 ≥ 0.6** 달성 확인

In [ ]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

# 문서 및 정답 데이터 (수정하지 마시오)
documents = [
    {"id": "doc_01", "text": "Annual leave requests must be submitted through the HR portal at least 3 business days in advance. Emergency leave requires manager approval via email within 24 hours of the absence."},
    {"id": "doc_02", "text": "Business expense reimbursement requires original receipts and a completed expense form submitted to Finance before the last working day of the month in which expenses were incurred."},
    {"id": "doc_03", "text": "Remote work arrangements must be approved by the direct line manager and formally registered with the People Operations team at least one week before the start date."},
    {"id": "doc_04", "text": "Base salary is disbursed on the 25th of each month. Performance bonuses are paid quarterly, calculated as a percentage of quarterly KPI achievement as assessed by the performance review committee."},
    {"id": "doc_05", "text": "Professional development funding of up to USD 1,500 per year is available to all full-time employees. Prior written approval from the Learning & Development team is required before enrolment."},
    {"id": "doc_06", "text": "Paid time off accrues based on years of service: 15 days for 0–2 years, 20 days for 3–5 years, and 25 days for 6+ years. Unused leave above 5 days does not carry over to the next calendar year."},
    {"id": "doc_07", "text": "Business travel expenses including flights, hotels, and ground transport should be booked through the corporate travel portal. Personal card purchases are reimbursable only when the portal is unavailable."},
    {"id": "doc_08", "text": "New employee onboarding spans the first two weeks of employment. Week 1 covers administrative setup and HR policy orientation; Week 2 includes team introductions and role-specific technical training."},
    {"id": "doc_09", "text": "Employee wellness benefits include a USD 600 annual wellness allowance redeemable at approved fitness and healthcare partners, plus access to the Employee Assistance Programme for confidential counselling."},
    {"id": "doc_10", "text": "Severance pay is calculated as one month of average base salary for each completed year of service, applicable to employees with at least one year of continuous employment at the time of termination."},
]

# 쿼리와 정답 문서 ID (수정하지 마시오)
qa_pairs = [
    {"query": "How do I submit a request for annual leave?",          "answer_id": "doc_01"},
    {"query": "What is the procedure for claiming business expenses?", "answer_id": "doc_02"},
    {"query": "What steps are required to work from home?",           "answer_id": "doc_03"},
    {"query": "When is the monthly salary paid?",                     "answer_id": "doc_04"},
    {"query": "How many days of paid leave am I entitled to?",        "answer_id": "doc_06"},
]

model = SentenceTransformer("all-MiniLM-L6-v2")

# TODO 1: ChromaDB에 문서 저장
# 컬렉션 이름: "q6_collection" 고정 (다른 문제와 충돌 방지)

# TODO 2: 실험 1 — k=1, 3, 5에 대해 Recall@k 계산 및 출력
print("[실험 1: k값 변경]")
for k in [1, 3, 5]:
    # TODO
    pass

# TODO 3: 실험 2 — threshold=0.3, 0.5, 0.7에서 평균 반환 문서 수 출력
# (k=10으로 검색 후, distance < threshold 조건으로 필터링)
print("\n[실험 2: score_threshold 필터링]")
for threshold in [0.3, 0.5, 0.7]:
    # TODO
    pass

---
# 문제 7. Cross-Encoder Re-ranking 구현 [Part 4-2] ★ 킬러문항

아래 조건을 만족하는 **2-Stage 검색 파이프라인**을 구현하시오.

- **Stage 1 (Bi-Encoder)**: `all-MiniLM-L6-v2`로 코사인 유사도 기준 Top-5 후보 선정
- **Stage 2 (Cross-Encoder)**: `cross-encoder/ms-marco-MiniLM-L-6-v2`로 Top-5 후보 재정렬

**필수 출력 항목**:
1. Stage 1 결과: 순위, 코사인 유사도, 문서 ID, 내용 앞 50자
2. Stage 2 결과: 순위, Cross-Encoder 점수, 문서 ID, 내용 앞 50자
3. 순위 변화 요약: 순위가 변한 문서에 대해 `"doc_XX: Stage1 rank N → Stage2 rank M"` 형식으로 출력

**⚠️ 킬러 채점 기준**:
- Stage 1, Stage 2 결과 모두 포함된 출력 **스크린샷 제출 필수**
- Stage 1 Top-1과 Stage 2 Top-1이 **서로 다른 문서**여야 정상 동작으로 인정

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

# 주어진 데이터 (수정하지 마시오)
query = "How to improve the accuracy of information retrieval in RAG systems?"

candidates = [
    {"id": "doc_01", "text": "Re-ranking with a Cross-Encoder significantly improves retrieval precision in RAG."},
    {"id": "doc_02", "text": "Vector databases store embeddings for fast approximate nearest neighbor search."},
    {"id": "doc_03", "text": "Hybrid search combining BM25 and dense retrieval improves recall in RAG pipelines."},
    {"id": "doc_04", "text": "Chunking strategy affects the quality of context provided to the language model."},
    {"id": "doc_05", "text": "Query expansion techniques help retrieve more relevant documents from the corpus."},
    {"id": "doc_06", "text": "The embedding model converts text into high-dimensional dense vectors for similarity search."},
    {"id": "doc_07", "text": "Metadata filtering allows targeted retrieval from specific document subsets."},
    {"id": "doc_08", "text": "Evaluating RAG systems requires metrics like faithfulness and answer relevancy from Ragas."},
]

# TODO 1: Stage 1 — Bi-Encoder로 전체 후보 임베딩 후 코사인 유사도 Top-5 선정
bi_encoder = 
# ... 코사인 유사도 계산 및 정렬

print("[Stage 1: Bi-Encoder Top-5 결과]")
print("-" * 65)
# TODO: 순위, 코사인 유사도, 문서 ID, 내용 앞 50자 출력

# TODO 2: Stage 2 — Cross-Encoder: Re-rank Top-5
cross_encoder = 
# cross_encoder.predict([(query, text), ...])으로 (쿼리, 문서) 쌍 점수 계산

print("\n[Stage 2: Cross-Encoder Re-ranking 결과 Top-5]")
print("-" * 65)
# TODO: 순위, Cross-Encoder 점수, 문서 ID, 내용 앞 50자 출력

# TODO 3: 순위 변화 요약 출력
print("\n[순위 변화 요약]")
# TODO: 순위가 바뀐 문서에 대해 "doc_XX: Stage1 rank N → Stage2 rank M" 형식으로 출력

---
# 문제 8. End-to-End QA 파이프라인 구축 [Part 5-1]

아래 조건을 만족하는 QA 파이프라인을 구현하시오.

- 주어진 문서들을 청킹 → 임베딩 → ChromaDB 저장
- LangChain의 `RetrievalQA` 또는 LCEL을 사용하여 QA 체인 구성
- **LLM**: `gemini-3.1-flash-lite-preview` (`ChatGoogleGenerativeAI` 사용)
- **임베딩**: `gemini-embedding-001` (`GoogleGenerativeAIEmbeddings` 사용)
- 아래 3개 질문에 대해 **답변**과 **출처 문서(`source_documents`)** 함께 출력

**채점 기준**: 3개 질문 모두 답변 생성 + 출처 문서 1개 이상 정상 반환

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 지식 베이스 문서 (수정하지 마시오)
raw_docs = [
    Document(
        page_content=(
            "RAG (Retrieval-Augmented Generation) is an AI architecture that grounds LLM outputs "
            "in dynamically retrieved external documents, mitigating the knowledge cutoff problem "
            "and reducing hallucination. Unlike fine-tuning, which updates model weights, RAG keeps "
            "the knowledge base external and updateable without retraining the model."
        ),
        metadata={"source": "intro.txt"}
    ),
    Document(
        page_content=(
            "Embedding models convert text into dense high-dimensional vectors such that semantically "
            "similar texts are geometrically close. gemini-embedding-001 produces 3072-dimensional "
            "vectors and consistently ranks among the top models on the MTEB multilingual benchmark. "
            "The model supports flexible output dimensionality via Matryoshka Representation Learning, "
            "allowing truncation to 768 or 1536 dimensions with minimal quality loss."
        ),
        metadata={"source": "embedding.txt"}
    ),
    Document(
        page_content=(
            "Chunking splits long documents into embedding-sized fragments. The two key parameters "
            "are chunk_size (max characters or tokens per chunk) and chunk_overlap (shared characters "
            "between adjacent chunks). A chunk_overlap of 10–20% of chunk_size prevents context loss "
            "at boundaries. RecursiveCharacterTextSplitter tries paragraph, sentence, and word "
            "boundaries in sequence, producing more natural splits than fixed-character splitting."
        ),
        metadata={"source": "chunking.txt"}
    ),
    Document(
        page_content=(
            "ChromaDB is an open-source Python-native vector database. It supports both ephemeral "
            "in-memory mode (ideal for prototyping) and persistent disk-based storage. ChromaDB "
            "integrates natively with LangChain via the Chroma vectorstore wrapper and supports "
            "metadata filtering to restrict search to document subsets. Collections are the primary "
            "organisational unit; each collection holds documents, embeddings, and metadata."
        ),
        metadata={"source": "vectordb.txt"}
    ),
    Document(
        page_content=(
            "Re-ranking improves retrieval precision by applying a Cross-Encoder model to the "
            "top-K candidates returned by Stage 1 Bi-Encoder search. A Cross-Encoder processes "
            "the query and document jointly, capturing token-level interactions that independent "
            "Bi-Encoder embeddings miss. The 2-Stage pipeline (Bi-Encoder → Top-K → Cross-Encoder → "
            "Top-N) balances the speed of approximate search with the accuracy of joint encoding."
        ),
        metadata={"source": "reranking.txt"}
    ),
    Document(
        page_content=(
            "Ragas is an evaluation framework for RAG systems providing four core metrics: "
            "Faithfulness (is every claim in the answer supported by the retrieved context?), "
            "Answer Relevancy (does the answer address the question without unnecessary content?), "
            "Context Precision (what fraction of retrieved chunks were necessary for the answer?), "
            "and Context Recall (were all chunks needed to construct the reference answer retrieved?). "
            "Scores range from 0 to 1; higher is better for all four metrics."
        ),
        metadata={"source": "evaluation.txt"}
    ),
]

questions = [
    "How does RAG reduce hallucination compared to standard LLM generation?",
    "What are the key features and use cases of ChromaDB?",
    "Why is re-ranking necessary and how does a Cross-Encoder differ from a Bi-Encoder?"
]

# TODO 1: 문서 청킹 (chunk_size=300, chunk_overlap=50)
splitter = 
chunks = 

# TODO 2: GoogleGenerativeAIEmbeddings로 ChromaDB 벡터스토어 생성
# 모델: "gemini-embedding-001"
embeddings = 
vectorstore = 

# TODO 3: LLM 설정 및 RetrievalQA 체인 구성
# 모델: "gemini-3.1-flash-lite-preview", return_source_documents=True
llm = 
qa_chain = 

# TODO 4: 3개 질문에 대해 답변 + 출처 출력
for question in questions:
    # TODO
    # 출력 형식:
    # Q: 질문
    # A: 답변
    # 출처: source 파일명
    # ---
    pass

---
# 문제 9. Ragas를 활용한 RAG 평가 [Part 6-1] 

아래 조건을 만족하도록 Ragas 평가를 수행하시오.

- 주어진 평가 데이터셋(5개 QA 쌍)을 사용하여 Ragas 평가 실행
- 측정 지표: `Faithfulness`, `AnswerRelevancy`, `ContextPrecision`, `ContextRecall` **4개 모두**
- Ragas 평가 시 LLM으로 `gemini-3.1-flash-lite-preview`를 사용하시오.
- 평가 결과를 **DataFrame**으로 출력하고, 지표별 평균값을 계산하여 출력

**채점 기준**:
- 4개 지표 모두 포함된 평가 결과 DataFrame **스크린샷 제출 필수**
- **Faithfulness 평균 ≥ 0.5** 달성 확인
  - 미달 시: 컨텍스트 또는 답변을 수정하고 **재실행 후 통과한 결과** 스크린샷 제출

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset
import pandas as pd

# 평가 데이터셋 (수정하지 마시오)
eval_data = {
    "question": [
        "What is a RAG system and what problem does it solve?",
        "What is the role of an embedding model in a RAG pipeline?",
        "Why is chunking necessary and what are its key parameters?",
        "What are the main features of ChromaDB?",
        "What is re-ranking and how does a Cross-Encoder work?",
    ],
    "answer": [
        "RAG (Retrieval-Augmented Generation) grounds LLM outputs in retrieved external documents, mitigating hallucination and the knowledge cutoff problem.",
        "An embedding model encodes text as dense vectors so that semantically similar texts are geometrically close, enabling semantic similarity search.",
        "Chunking splits long documents into embedding-sized fragments. The key parameters are chunk_size (max fragment length) and chunk_overlap (shared text between adjacent chunks).",
        "ChromaDB is an open-source Python-native vector database supporting in-memory and persistent modes, with native LangChain integration and metadata filtering.",
        "Re-ranking reorders the top-K candidates from Stage 1 retrieval using a Cross-Encoder that scores query-document pairs jointly, improving precision over independent Bi-Encoder embeddings.",
    ],
    "contexts": [
        [
            "RAG (Retrieval-Augmented Generation) is an AI architecture that grounds LLM outputs in dynamically retrieved external documents, mitigating the knowledge cutoff problem and reducing hallucination. Unlike fine-tuning, which updates model weights, RAG keeps the knowledge base external and updateable without retraining the model."
        ],
        [
            "Embedding models convert text into dense high-dimensional vectors such that semantically similar texts are geometrically close. gemini-embedding-001 produces 3072-dimensional vectors and consistently ranks among the top models on the MTEB multilingual benchmark."
        ],
        [
            "Chunking splits long documents into embedding-sized fragments. The two key parameters are chunk_size (max characters or tokens per chunk) and chunk_overlap (shared characters between adjacent chunks). A chunk_overlap of 10–20% of chunk_size prevents context loss at boundaries."
        ],
        [
            "ChromaDB is an open-source Python-native vector database. It supports both ephemeral in-memory mode and persistent disk-based storage. ChromaDB integrates natively with LangChain and supports metadata filtering to restrict search to document subsets."
        ],
        [
            "Re-ranking improves retrieval precision by applying a Cross-Encoder model to the top-K candidates returned by Stage 1 Bi-Encoder search. A Cross-Encoder processes the query and document jointly, capturing token-level interactions that independent Bi-Encoder embeddings miss."
        ],
    ],
    "reference": [
        "RAG is a framework that retrieves relevant external documents at inference time to ground LLM generation, addressing knowledge staleness and hallucination.",
        "Embedding models map text to high-dimensional dense vectors enabling semantic similarity search based on cosine distance.",
        "Chunking divides documents into smaller fragments suitable for embedding; chunk_size and chunk_overlap are the primary hyperparameters controlling fragment size and boundary overlap.",
        "ChromaDB is an open-source vector database with Python-native API, in-memory and persistent storage options, and built-in LangChain integration.",
        "Re-ranking uses a Cross-Encoder that jointly encodes query and document to produce accurate relevance scores, improving precision over Bi-Encoder-based first-stage retrieval.",
    ]
}

# TODO 1: Dataset 객체 생성
dataset = 

# TODO 2: Ragas용 LLM 및 Embeddings 설정 (gemini-3.1-flash-lite-preview, gemini-embedding-001)
ragas_llm = 
ragas_embeddings = 

# TODO 3: 각 메트릭에 llm, embeddings 설정
# 예: faithfulness.llm = ragas_llm

# TODO 4: Ragas evaluate 실행 (4개 지표 모두 포함)
result = 

# TODO 5: 결과 DataFrame 출력

# TODO 6: 지표별 평균값 및 PASS/FAIL 출력
# 출력 형식:
# === 평가 결과 요약 ===
# Faithfulness:      0.XXXX
# AnswerRelevancy:   0.XXXX
# ContextPrecision:  0.XXXX
# ContextRecall:     0.XXXX
# Faithfulness >= 0.5 달성 여부: PASS / FAIL

---
# 문제 10. RAG 파이프라인 아키텍처 설계 [Part 7] 

아래 요구사항을 만족하는 **완전한 RAG 서비스 아키텍처**를 설계하고 각 컴포넌트를 코드로 정의하시오.

**필수 컴포넌트 6개 (모두 포함)**:

| 컴포넌트 | 역할 |
|---------|------|
| `DocumentLoader` | 다양한 형식 파싱 → `List[Document]` 반환 |
| `TextProcessor` | 청킹 전략 적용 → `List[Document]` (청크 리스트) 반환 |
| `EmbeddingService` | 임베딩 모델 관리 및 벡터 생성 |
| `VectorStore` | 벡터 DB 저장/검색 인터페이스 |
| `Retriever` | Bi-Encoder 검색 + Cross-Encoder Re-ranking |
| `RAGPipeline` | 전체 파이프라인 통합 (질문 → 답변) |

**코드 요구사항**:
- 각 컴포넌트를 Python 클래스로 정의 (메서드 시그니처 및 docstring 필수)
- `RAGPipeline.query(question: str)` → `{"answer": str, "sources": list, "scores": list}` 형태로 반환
- 실제 실행 가능한 코드일 필요는 없으나, **설계 의도가 명확히 드러나는 구조**여야 함

**채점 기준**:
- 6개 컴포넌트 클래스 모두 정의 및 하단 검증 코드 정상 출력 **스크린샷 제출 필수**
- 아래 마크다운 셀에 **Indexing / Querying 흐름 다이어그램 직접 작성** 필수 (ASCII 아트 또는 화살표 표기)
- 다이어그램 미작성 시 0점 처리

In [ ]:
from typing import List, Dict, Any
from langchain_core.documents import Document

# TODO: 6개 컴포넌트 클래스 구현

class DocumentLoader:
    """다양한 형식의 문서를 파싱하여 Document 객체 리스트로 반환"""

    def load(self, file_path: str) -> List[Document]:
        # TODO: 문제 2의 parse_document() 로직 참고
        pass


class TextProcessor:
    """Document 리스트를 청킹하여 Chunk 리스트로 반환"""

    def __init__(self, chunk_size: int = 300, chunk_overlap: int = 50):
        # TODO
        pass

    def process(self, documents: List[Document]) -> List[Document]:
        # TODO
        pass


class EmbeddingService:
    """임베딩 모델을 관리하고 텍스트 벡터를 생성"""

    def __init__(self, model_name: str = "gemini-embedding-001"):
        # TODO
        pass

    def embed(self, texts: List[str]) -> List[List[float]]:
        # TODO
        pass


class VectorStore:
    """벡터 DB 저장 및 검색 인터페이스 (ChromaDB 기반)"""

    def __init__(self, embedding_service: EmbeddingService):
        # TODO
        pass

    def add_documents(self, documents: List[Document]) -> None:
        # TODO
        pass

    def search(self, query: str, k: int = 10) -> List[Document]:
        # TODO
        pass


class Retriever:
    """Bi-Encoder 검색 후 Cross-Encoder Re-ranking 수행 (2-Stage)"""

    def __init__(self, vector_store: VectorStore, top_k: int = 5, rerank_top_n: int = 3):
        # TODO
        pass

    def retrieve(self, query: str) -> List[Document]:
        """Stage 1: Bi-Encoder 검색 (top_k) → Stage 2: Cross-Encoder Re-ranking (rerank_top_n)"""
        # TODO
        pass


class RAGPipeline:
    """전체 RAG 파이프라인 통합: 문서 인덱싱 및 질문-답변"""

    def __init__(self, retriever: Retriever, llm_model: str = "gemini-3.1-flash-lite-preview"):
        # TODO
        pass

    def index(self, file_paths: List[str]) -> None:
        """문서 인덱싱: 로드 → 청킹 → 임베딩 → 저장"""
        # TODO
        pass

    def query(self, question: str) -> Dict[str, Any]:
        """
        질문에 대한 RAG 답변 생성

        Returns:
            {
                "answer": str,      # LLM 생성 답변
                "sources": list,    # 출처 Document 리스트
                "scores": list      # Re-ranking 점수 리스트
            }
        """
        # TODO
        pass


# 검증 코드 (수정하지 마시오)
classes = [DocumentLoader, TextProcessor, EmbeddingService, VectorStore, Retriever, RAGPipeline]
print("6개 컴포넌트 클래스 정의 완료")
for cls in classes:
    print(f"  - {cls.__name__}: {cls.__doc__.strip()}")

### 📐 RAG 파이프라인 데이터 흐름 다이어그램 (직접 작성 필수)

> Indexing 흐름과 Querying 흐름을 구분하여 ASCII 아트 또는 텍스트 화살표로 작성하시오.
> 각 화살표에 데이터 형태(예: `List[Document]`, `List[float]`)를 표기하면 가점.

**[Indexing 흐름]**
```
(여기에 작성)
예시: 문서 파일 → DocumentLoader → TextProcessor → EmbeddingService → VectorStore
```

**[Querying 흐름]**
```
(여기에 작성)
예시: 사용자 질문 → Retriever(Stage1→Stage2) → RAGPipeline → LLM → 답변+출처
```